Imports

In [ ]:
#import sys
!{sys.executable} -m pip install -r ../requirements.txt

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached langchain_openai-1.2.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_pinecone-0.2.13-py3-none-any.whl.metadata (8.6 kB)
  Using cached pinecone-9.0.0-cp310-abi3-macosx_10_12_x86_64.whl.metadata (5.0 kB)
  Using cached langchain_protocol-0.0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_10_13_x86_64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached xxhash-3.7.0-cp313-cp313-macosx_10_13_x86_64.whl.metadata (13 kB)
  Using cached ormsgpack-1.12.2-cp313-cp313-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore

from pinecone import Pinecone, ServerlessSpec

Load environment variables

In [2]:
load_dotenv("../.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "mental-space-ai")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

print("OpenAI key loaded:", OPENAI_API_KEY is not None)
print("Pinecone key loaded:", PINECONE_API_KEY is not None)
print("Pinecone index:", PINECONE_INDEX_NAME)

OpenAI key loaded: True
Pinecone key loaded: True
Pinecone index: mental-space-ai


testing the flow for now

In [3]:
source_folder = Path("../data/source_documents")
source_folder.mkdir(parents=True, exist_ok=True)

sample_file = source_folder / "sample_msp_text.txt"

sample_file.write_text(
    """
Mental Space Psychology works with the idea that thoughts, emotions, relationships, goals, and identity are represented spatially in the mind.

When someone feels unclear, overwhelmed, or stuck, it may be because their internal mental structure is disorganized.

Mental Space Coaching helps make these inner representations visible and reorganizes them so the person can experience more clarity, direction, and inner stability.
""",
    encoding="utf-8"
)

print("Sample document created:", sample_file)

Sample document created: ../data/source_documents/sample_msp_text.txt


Document loading

In [4]:
def load_documents(folder_path):
    folder = Path(folder_path)
    documents = []

    for file_path in folder.iterdir():
        if file_path.is_file():
            suffix = file_path.suffix.lower()

            if suffix == ".txt":
                loader = TextLoader(str(file_path), encoding="utf-8")
                documents.extend(loader.load())

            elif suffix == ".pdf":
                loader = PyPDFLoader(str(file_path))
                documents.extend(loader.load())

            elif suffix == ".docx":
                loader = Docx2txtLoader(str(file_path))
                documents.extend(loader.load())

            else:
                print("Skipping unsupported file:", file_path.name)

    return documents


documents = load_documents("../data/source_documents")

print("Documents loaded:", len(documents))
print(documents[0].page_content[:500])

Documents loaded: 1

Mental Space Psychology works with the idea that thoughts, emotions, relationships, goals, and identity are represented spatially in the mind.

When someone feels unclear, overwhelmed, or stuck, it may be because their internal mental structure is disorganized.

Mental Space Coaching helps make these inner representations visible and reorganizes them so the person can experience more clarity, direction, and inner stability.



Chunking

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"chunk_{i}"
    chunk.metadata["source_type"] = "local_document"

print("Chunks created:", len(chunks))
print(chunks[0].page_content)
print(chunks[0].metadata)

Chunks created: 1
Mental Space Psychology works with the idea that thoughts, emotions, relationships, goals, and identity are represented spatially in the mind.

When someone feels unclear, overwhelmed, or stuck, it may be because their internal mental structure is disorganized.

Mental Space Coaching helps make these inner representations visible and reorganizes them so the person can experience more clarity, direction, and inner stability.
{'source': '../data/source_documents/sample_msp_text.txt', 'chunk_id': 'chunk_0', 'source_type': 'local_document'}


Creating Pinecone Index

In [6]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [index["name"] for index in pc.list_indexes()]

if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION
        )
    )
    print("Created Pinecone index:", PINECONE_INDEX_NAME)
else:
    print("Pinecone index already exists:", PINECONE_INDEX_NAME)

Created Pinecone index: mental-space-ai


Create embeddings and vector store

In [7]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)

print("Vector store connected.")

Vector store connected.


Add chunks to Pinecone

In [8]:
vectorstore.add_documents(chunks)

print("Chunks uploaded to Pinecone.")

Chunks uploaded to Pinecone.


Retrieval test

In [9]:
query = "Why does clarity not come from more thinking but from reorganizing mental space?"

retrieved_docs = vectorstore.similarity_search(
    query=query,
    k=3
)

print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print(doc.metadata)

Retrieved documents: 1

--- Result 1 ---
Mental Space Psychology works with the idea that thoughts, emotions, relationships, goals, and identity are represented spatially in the mind.

When someone feels unclear, overwhelmed, or stuck, it may be because their internal mental structure is disorganized.

Mental Space Coaching helps make these inner representations visible and reorganizes them so the person can experience more clarity, direction, and inner stability.
{'chunk_id': 'chunk_0', 'source': '../data/source_documents/sample_msp_text.txt', 'source_type': 'local_document'}


test

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3
)

context = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
You are the first local prototype of a Mental Space Psychology AI content agent.

Use only the source context below.

Topic:
{query}

Source context:
{context}

Create:
1. Short expert summary
2. Public-friendly explanation
3. Instagram caption draft
4. Three hooks
5. SEO keywords
6. Hashtags
7. Linkedin post adapted for Linkedin audiences
8. One simple, safe reflection exercise

Important:
- Do not make exaggerated healing claims.
- Do not suggest trauma work.
- Keep it clear, grounded, and human.
- If the source context is not enough, say so.
"""

response = llm.invoke(prompt)

print(response.content)

### 1. Short Expert Summary
Clarity in thought and emotion does not stem from excessive rumination but rather from reorganizing one's mental space. Mental Space Psychology posits that our thoughts, feelings, and identities are spatially represented in our minds. When individuals feel overwhelmed or unclear, it often indicates a disorganized internal structure. Mental Space Coaching aims to visualize and reorganize these inner representations, leading to enhanced clarity, direction, and stability.

### 2. Public-Friendly Explanation
When we feel confused or stuck, it’s not always because we need to think harder. Instead, it might be that our thoughts and feelings are jumbled up in our minds. Mental Space Psychology helps us see and rearrange these mental "spaces," making it easier to find clarity and direction. By organizing our inner world, we can feel more stable and focused.

### 3. Instagram Caption Draft
✨ Feeling overwhelmed or unclear? Sometimes, clarity doesn’t come from thinkin

Reusable function

In [11]:
def ask_msp_agent(topic, top_k=5):
    retrieved_docs = vectorstore.similarity_search(topic, k=top_k)

    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
You are a Mental Space Psychology AI content assistant.

Use only the retrieved source context.

Topic:
{topic}

Source context:
{context}

Create:
1. Short expert summary
2. Public-friendly explanation
3. Instagram caption draft
4. LinkedIn post draft adapted for Linkedin audiences
5. Five carousel slide ideas
6. Three hooks
7. SEO keywords
8. Hashtags
9. CTA
10. One simple, safe reflection exercise

Rules:
- No exaggerated claims.
- No trauma intervention.
- Keep the language accessible and professional.
"""

    response = llm.invoke(prompt)
    return response.content

changing topic structure

In [12]:
import random

topics = [
    "Why clarity does not come from more thinking",
    "How mental space affects decision-making",
    "Why people feel stuck during transitions",
    "How internal structure influences leadership presence",
    "How relationship patterns can be represented in mental space"
]

def get_next_topic():
    return random.choice(topics)

topic = get_next_topic()

print("Selected topic:")
print(topic)

result = ask_msp_agent(topic)

print(result)

Selected topic:
How relationship patterns can be represented in mental space
### 1. Short Expert Summary
Mental Space Psychology posits that our thoughts, emotions, relationships, goals, and identities are spatially represented in our minds. Disorganization in these mental structures can lead to feelings of confusion or being stuck. Mental Space Coaching aims to clarify and reorganize these inner representations, fostering greater clarity, direction, and stability.

### 2. Public-Friendly Explanation
Have you ever felt overwhelmed or unsure about your relationships or goals? Mental Space Psychology suggests that our thoughts and feelings are like a map in our minds. When this map gets messy, it can be hard to find your way. Mental Space Coaching helps you tidy up that map, making it easier to see where you want to go and how to get there.

### 3. Instagram Caption Draft
🌌 Feeling lost in your thoughts or relationships? Mental Space Psychology can help! By organizing the way we mentally